# Day 1 From GPT logits to a trained assistant

Week 1 exposed the computation. Today we compare checkpoints with different training histories.

**Budget:** 45 minutes. **Before you start:** tokens, attention and next-token logits.

Run the cells in order in Colab. Model cells are tagged `live` and download a public checkpoint. Deterministic exercises are labelled separately and use no model. Outputs start empty: record your own observations. Never report a fixture as a model result.


In [1]:
import json, math, re, time, html, statistics
from pathlib import Path
try:
    from IPython.display import display, HTML
except ImportError:
    def HTML(s): return s
    def display(s): print(s)

def show_table(headers, rows):
    def cell(v): return html.escape(str(v))
    header = ''.join('<th style="padding:8px;border:1px solid #ccc">'+cell(h)+'</th>' for h in headers)
    body = ''.join('<tr>'+''.join('<td style="padding:8px;border:1px solid #ccc">'+cell(v)+'</td>' for v in row)+'</tr>' for row in rows)
    display(HTML('<table style="border-collapse:collapse"><tr>'+header+'</tr>'+body+'</table>'))

def bars(labels, values, title):
    rows = []
    for label,value in zip(labels,values):
        rows.append('<div style="display:flex;align-items:center;gap:10px;margin:5px 0">'
                    '<span style="width:120px">'+html.escape(str(label))+'</span>'
                    '<span style="display:inline-block;background:#377da8;height:16px;width:'+str(max(0, value)*320)+'px"></span>'
                    '<span>'+format(value,'.3f')+'</span></div>')
    display(HTML('<h4>'+html.escape(title)+'</h4>'+''.join(rows)))


## 1 Identify the training target
This is a word-level teaching illustration, not the tokenizer used by Qwen. Create input/next-token pairs and explain where the label comes from.


In [2]:
words='the club meets on Friday'.split()
show_table(['Input prefix','Next token target'], [(' '.join(words[:i]),words[i]) for i in range(1,len(words))])


Input prefix,Next token target
the,club
the club,meets
the club meets,on
the club meets on,Friday


**Write:** Why is next-token prediction called self-supervised? Which tokens are visible when the model predicts `Friday`?


## Trace the lifecycle
Sketch text data → next-token training → base checkpoint → supervised demonstrations and/or preference training → assistant checkpoint → inference. Label which stages update weights and which do not. Keep this sketch alongside the checkpoint comparison below.


## Model setup
The first download may take extra time. A free GPU is optional; CPU inference is slower. The teaching model is small, so errors are expected and useful to inspect. Setup does not require an API key.


In [4]:
# This downloads public model files. No paid API or account key is used.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers>=4.49,<5', 'torch>=2.2', 'pillow'])
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
print('torch', torch.__version__)
import transformers
from importlib.metadata import version
if transformers.__version__ != version('transformers'):
    raise RuntimeError('A previously imported version is still in memory. Restart the session and rerun this notebook.')
print('transformers', transformers.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
tok, model, active_model = None, None, None

def load_model(model_id='Qwen/Qwen2.5-0.5B-Instruct'):
    global tok, model, active_model
    if active_model == model_id and model is not None:
        return
    # Release the previous checkpoint before downloading/loading the next one.
    model = None
    import gc
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=DTYPE).to(DEVICE).eval()
    active_model = model_id
    print('Loaded', model_id, 'on', DEVICE)

def prepare(text, system=None, chat=True):
    if model is None: raise RuntimeError('Run load_model() first.')
    if chat:
        messages = []
        if system: messages.append({'role':'system','content':system})
        messages.append({'role':'user','content':text})
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    batch = tok(text, return_tensors='pt', add_special_tokens=False).to(DEVICE)
    if batch['input_ids'].shape[1] > 1024:
        raise ValueError('This teaching notebook caps inputs at 1024 tokens. Shorten the input explicitly.')
    return batch

def generate(text, system=None, temperature=0, top_k=0, top_p=1.0, max_new_tokens=96, chat=True):
    if temperature < 0: raise ValueError('temperature must be nonnegative')
    batch = prepare(text, system, chat)
    sampling = dict(do_sample=False)
    if temperature > 0:
        sampling = dict(do_sample=True, temperature=temperature, top_k=top_k, top_p=top_p)
    start = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**batch, max_new_tokens=max_new_tokens,
                             pad_token_id=tok.eos_token_id, **sampling)
    raw = tok.decode(out[0,batch['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return {'raw':raw, 'seconds':time.perf_counter()-start,
            'input_tokens':batch['input_ids'].shape[1],
            'output_tokens':out.shape[1]-batch['input_ids'].shape[1], 'model':active_model}

load_model()


torch 2.11.0+cpu
transformers 4.57.6
Loaded Qwen/Qwen2.5-0.5B-Instruct on cpu


## 2 Inspect the message format
Roles are serialised into tokens. The template is part of the model interface; adding role labels manually is not equivalent.


In [5]:
messages=[{'role':'system','content':'Answer concisely.'},{'role':'user','content':'What is a context window?'}]
print(tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True))


<|im_start|>system
Answer concisely.<|im_end|>
<|im_start|>user
What is a context window?<|im_end|>
<|im_start|>assistant



## 3 Compare related checkpoints
The instruct model gets its supported chat template. The base model gets raw text. This compares appropriate use of two checkpoints; it does not isolate a single training intervention.


In [6]:
prompts=['The best thing about a study group is', 'What is a context window?', 'Explain attention in exactly two short bullet points.']
results={}
for model_id,is_chat in [('Qwen/Qwen2.5-0.5B',False),('Qwen/Qwen2.5-0.5B-Instruct',True)]:
    load_model(model_id)
    results[model_id]=[generate(p,chat=is_chat,max_new_tokens=64) for p in prompts]
show_table(['Prompt','Base output','Instruct output'],[(p,results['Qwen/Qwen2.5-0.5B'][i]['raw'],results['Qwen/Qwen2.5-0.5B-Instruct'][i]['raw']) for i,p in enumerate(prompts)])


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-0.5B on cpu


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loaded Qwen/Qwen2.5-0.5B-Instruct on cpu


Prompt,Base output,Instruct output
The best thing about a study group is,"that you can learn from each other. But it can also be a bit of a burden. You may not be able to concentrate on your own work, and you may find that you are constantly interrupted by other people. This is because you are in a group, and you are all working towards the same goal. However",A study group can be incredibly beneficial for several reasons: 1. **Enhanced Collaboration**: Studying with peers allows you to work together on projects and share knowledge. This collaborative environment can help you learn from each other's perspectives and improve your understanding of the material. 2. **Increased Knowledge**: Working in groups often leads
What is a context window?,"A context window is a window that displays information about the current context of a program. It is typically used in graphical user interfaces (GUIs) to provide information about the current window, such as the size, position, and color of the window. Context windows are often used to display information about the current window, such",A context window in the context of AI and machine learning refers to a portion of data that is used for training models or analyzing patterns within a larger dataset. It's typically a small segment of information that helps the model learn from its interactions with this data. Context windows can be useful for several reasons: 1. **Data
Explain attention in exactly two short bullet points.,"1. Attention is the process of focusing one's attention on a particular object, idea, or activity. 2. Attention is a cognitive process that involves the ability to concentrate on a particular stimulus or object for a specific period of time.",Attention is an important component of neural networks that enables the model to focus on relevant parts of the input data during processing. It works by projecting the input data into higher-dimensional feature spaces and then selectively selecting the most relevant features for further processing based on their importance or relevance. This allows the model to learn more efficiently and effectively


## 4 Model card check
Read the [base card](https://huggingface.co/Qwen/Qwen2.5-0.5B) and [instruct card](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct). Record the licence, model-specific context length and one relevant limitation. Distinguish a family-wide headline from the exact 0.5B configuration.

**Your observations:**
- Completion prompt:
- Question:
- Constrained request:
- What changed in training? What changed in input formatting?

**Exit check:** Does a base model always fail at questions? No; describe behaviour you actually observed rather than forcing the examples to match a slogan.
